# PD Dataset Construction

**Objective**

This notebook constructs the datasets required for Probability of Default (PD) model development from the cleaned and engineered Master Dataset.

The workflow:

1. defines the PD target and candidate predictors;
2. removes identifiers and post-origination leakage;
3. creates development and test samples;
4. applies preprocessing parameters learned exclusively from the development sample;
5. performs PD-specific feature screening and categorical regrouping;
6. produces native and fully numerical modelling datasets;
7. validates and exports the final datasets.

The Master Dataset remains unchanged. All transformations introduced here are specific to PD modelling.

In [207]:
# =============================================================================
# Imports and Configuration
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

In [208]:
# =============================================================================
# Project Paths
# =============================================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

MASTER_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "lending_club_master_dataset.parquet"
)

PD_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pd"
)

PD_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not MASTER_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Master Dataset not found: {MASTER_DATA_PATH}"
    )

In [209]:
# =============================================================================
# Load Master Dataset
# =============================================================================

master_data = pd.read_parquet(
    MASTER_DATA_PATH
)

print(f"Rows    : {master_data.shape[0]:,}")
print(f"Columns : {master_data.shape[1]}")

display(master_data.head())

Rows    : 466,285
Columns : 68


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,emp_length_years,emp_length_missing,credit_history_months,funding_ratio,investor_funding_ratio,investor_loan_ratio,loan_to_income_ratio,installment_to_income_ratio,open_account_ratio,credit_inquiry_rate,delinquency_rate,emp_title_missing,loan_burden_interest,mths_since_last_record_is_missing,mths_since_last_major_derog_is_missing,mths_since_last_delinq_is_missing
0,1077501,1296599,5000,5000,"4,975.0000",36,10.6500,162.8700,B,B2,NaN,10+ years,RENT,"24,000.0000",Verified,2011-12-01,Fully Paid,credit_card,Computer,860xx,AZ,27.6500,0.0000,1985-01-01,1.0000,-1.0000,-1.0000,3.0000,0.0000,13648,83.7000,9.0000,f,0.0000,0.0000,"5,861.0714","5,831.7800","5,000.0000",861.0700,0.0000,0.0000,0.0000,2015-01-01,171.6200,NaT,2016-01-01,0.0000,-1.0000,0.0000,NaN,NaN,NaN,10,0,323,1.0000,0.9950,0.9950,0.2083,0.0814,0.3333,0.0372,0.0000,1,2.2188,1,1,1
1,1077430,1314167,2500,2500,"2,500.0000",60,15.2700,59.8300,C,C4,Ryder,< 1 year,RENT,"30,000.0000",Source Verified,2011-12-01,Charged Off,car,bike,309xx,GA,1.0000,0.0000,1999-04-01,5.0000,-1.0000,-1.0000,3.0000,0.0000,1687,9.4000,4.0000,f,0.0000,0.0000,"1,008.7100","1,008.7100",456.4600,435.1700,0.0000,117.0800,1.1100,2013-04-01,119.6600,NaT,2013-09-01,0.0000,-1.0000,0.0000,NaN,NaN,NaN,0,0,152,1.0000,1.0000,1.0000,0.0833,0.0239,0.7500,0.3947,0.0000,0,1.2725,1,1,1
2,1077175,1313524,2400,2400,"2,400.0000",36,15.9600,84.3300,C,C5,NaN,10+ years,RENT,"12,252.0000",Not Verified,2011-12-01,Fully Paid,small_business,real estate business,606xx,IL,8.7200,0.0000,2001-11-01,2.0000,-1.0000,-1.0000,2.0000,0.0000,2956,98.5000,10.0000,f,0.0000,0.0000,"3,003.6536","3,003.6500","2,400.0000",603.6500,0.0000,0.0000,0.0000,2014-06-01,649.9100,NaT,2016-01-01,0.0000,-1.0000,0.0000,NaN,NaN,NaN,10,0,121,1.0000,1.0000,1.0000,0.1959,0.0826,0.2000,0.1983,0.0000,1,3.1263,1,1,1
3,1076863,1277178,10000,10000,"10,000.0000",36,13.4900,339.3100,C,C1,AIR RESOURCES BOARD,10+ years,RENT,"49,200.0000",Source Verified,2011-12-01,Fully Paid,other,personel,917xx,CA,20.0000,0.0000,1996-02-01,1.0000,35.0000,-1.0000,10.0000,0.0000,5598,21.0000,37.0000,f,0.0000,0.0000,"12,226.3022","12,226.3000","10,000.0000","2,209.3300",16.9700,0.0000,0.0000,2015-01-01,357.4800,NaT,2015-01-01,0.0000,-1.0000,0.0000,NaN,NaN,NaN,10,0,190,1.0000,1.0000,1.0000,0.2033,0.0828,0.2703,0.0632,0.0000,0,2.7419,1,1,0
4,1075358,1311748,3000,3000,"3,000.0000",60,12.6900,67.7900,B,B5,University Medical Group,1 year,RENT,"80,000.0000",Source Verified,2011-12-01,Current,other,Personal,972xx,OR,17.9400,0.0000,1996-01-01,0.0000,38.0000,-1.0000,15.0000,0.0000,27783,53.9000,38.0000,f,766.9000,766.9000,"3,242.1700","3,242.1700","2,233.1000","1,009.0700",0.0000,0.0000,0.0000,2016-01-01,67.7900,2016-02-01,2016-01-01,0.0000,-1.0000,0.0000,NaN,NaN,NaN,1,0,191,1.0000,1.0000,1.0000,0.0375,0.0102,0.3947,0.0000,0.0000,0,0.4759,1,1,0


# Define the PD Target

This section constructs the binary target variable used for Probability of Default (PD) modelling.

The target definition follows the business classification established for this project:

- `pd_target = 1`: the loan is classified as default;
- `pd_target = 0`: the loan is classified as non-default.

The following loan statuses are classified as default:

- `Charged Off`;
- `Default`;
- `Does not meet the credit policy. Status:Charged Off`;
- `Late (31-120 days)`.

All remaining loan statuses are classified as non-default under the methodology adopted for this project.

This target definition is applied consistently across all subsequent PD analyses and models.

In [210]:
# =============================================================================
# Audit the available loan statuses
# =============================================================================

pd_data = master_data.copy()

loan_status_distribution = (
    pd_data["loan_status"]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .reset_index(name="count")
)

loan_status_distribution["percentage"] = (
    loan_status_distribution["count"]
    .div(len(pd_data))
    .mul(100)
    .round(4)
)

display(loan_status_distribution)

,loan_status,count,percentage
0,Current,224226,48.0878
1,Fully Paid,184739,39.6193
2,Charged Off,42475,9.1092
3,Late (31-120 days),6900,1.4798
4,In Grace Period,3146,0.6747
5,Does not meet the credit policy. Status:Fully ...,1988,0.4263
6,Late (16-30 days),1218,0.2612
7,Default,832,0.1784
8,Does not meet the credit policy. Status:Charge...,761,0.1632


In [211]:
# =============================================================================
# Create and validate the PD target
# =============================================================================

DEFAULT_STATUSES = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Late (31-120 days)",
]

pd_data["pd_target"] = (
    pd_data["loan_status"]
    .isin(DEFAULT_STATUSES)
    .astype("int8")
)

status_target_validation = pd.crosstab(
    pd_data["loan_status"],
    pd_data["pd_target"],
    margins=True,
)

assert pd_data["pd_target"].isna().sum() == 0
assert set(pd_data["pd_target"].unique()) == {0, 1}

display(status_target_validation)

pd_target,0,1,All
loan_status,,,
Charged Off,0,42475,42475
Current,224226,0,224226
Default,0,832,832
Does not meet the credit policy. Status:Charged Off,0,761,761
Does not meet the credit policy. Status:Fully Paid,1988,0,1988
Fully Paid,184739,0,184739
In Grace Period,3146,0,3146
Late (16-30 days),1218,0,1218
Late (31-120 days),0,6900,6900


# Candidate Predictors

This section defines the initial set of candidate predictors for PD modelling.

Variables that directly identify borrowers or loans, as well as variables containing post-origination information or target leakage, are excluded before any preprocessing or feature selection.

The remaining variables constitute the initial predictor set for the subsequent modelling pipeline.

In [212]:
# =============================================================================
# 3. Define the Candidate Predictors
# =============================================================================

if "pd_target" not in pd_data.columns:
    raise KeyError(
        "The target variable 'pd_target' is missing from pd_data."
    )


NON_PREDICTOR_COLUMNS = [
    # Technical identifiers
    "id",
    "member_id",

    # Target and observed outcome
    "loan_status",
    "pd_target",

    # Post-origination repayment information
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
]

columns_removed = [
    column
    for column in NON_PREDICTOR_COLUMNS
    if column in pd_data.columns
]

X = pd_data.drop(
    columns=columns_removed
).copy()

y = pd_data["pd_target"].copy()


# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

remaining_non_predictors = [
    column
    for column in NON_PREDICTOR_COLUMNS
    if column in X.columns
]

if remaining_non_predictors:
    raise ValueError(
        "Non-predictor or leakage variables remain in X: "
        f"{remaining_non_predictors}"
    )

assert X.index.equals(y.index), (
    "The predictor matrix and target vector are not aligned."
)

assert len(X) == len(y), (
    "The predictor matrix and target vector have different lengths."
)

print(f"Observations         : {len(X):,}")
print(f"Candidate predictors : {X.shape[1]}")
print(f"Columns removed      : {len(columns_removed)}")

Observations         : 466,285
Candidate predictors : 52
Columns removed      : 17


# Create the Development and Test Samples

The candidate predictors are divided into development and test samples before any statistical preprocessing is performed.

The development sample is used to estimate all preprocessing parameters, including missing-value imputations, categorical regrouping rules and feature-selection decisions.

The test sample remains untouched throughout the data preparation process and is used only for the final evaluation of the PD models.

A stratified random split is performed to preserve the proportion of defaulted and non-defaulted loans in both samples.

In [213]:
# =============================================================================
# 4. Create the Development and Test Samples
# =============================================================================

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

split_summary = pd.DataFrame({
    "Observations": [
        len(X_train),
        len(X_test),
    ],
    "Default Rate (%)": [
        y_train.mean() * 100,
        y_test.mean() * 100,
    ],
}, index=[
    "Development sample",
    "Test sample",
])

split_summary["Default Rate (%)"] = (
    split_summary["Default Rate (%)"]
    .round(2)
)

display(split_summary)

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

assert len(X_train) + len(X_test) == len(X)

,Observations,Default Rate (%)
Development sample,373028,10.9300
Test sample,93257,10.9300


# Missing Value Analysis and Treatment

Missing values are treated only after the development and test samples have been created.

All statistical imputation parameters are estimated exclusively from the development sample (`X_train`) and then applied unchanged to the test sample (`X_test`) to prevent information leakage.

The treatment strategy is as follows:

- intentionally preserved missingness remains unchanged;
- numerical variables are imputed using the development-sample median;
- categorical variables are imputed using the development-sample mode;
- datetime variables are imputed using the development-sample median date.

The original variable `emp_length` intentionally preserves its missing values because this information is already represented by `emp_length_years` and `emp_length_missing`. The original variable is removed later during final dataset construction.

In [214]:
# =============================================================================
# Audit Missing Values in the Development Sample
# =============================================================================

missing_summary = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_percentage": (
        X_train.isna().mean().mul(100).round(2)
    ),
    "data_type": X_train.dtypes.astype(str),
})

missing_summary = (
    missing_summary
    .query("missing_count > 0")
    .sort_values(
        "missing_percentage",
        ascending=False,
    )
)

display(missing_summary)

,missing_count,missing_percentage,data_type
total_rev_hi_lim,56199,15.0700,float64
tot_cur_bal,56199,15.0700,float64
tot_coll_amt,56199,15.0700,float64
emp_title,21956,5.8900,str
emp_length,16715,4.4800,category
revol_util,280,0.0800,float64
collections_12_mths_ex_med,117,0.0300,float64
acc_now_delinq,22,0.0100,float64
credit_history_months,22,0.0100,Int16
earliest_cr_line,22,0.0100,datetime64[us]


In [215]:
# =============================================================================
# 5. Missing Value Analysis and Treatment
# =============================================================================

X_train = X_train.copy()
X_test = X_test.copy()

# -----------------------------------------------------------------------------
# 1. Identify variables automatically by data type
# -----------------------------------------------------------------------------

numeric_columns = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_columns = (
    X_train
    .select_dtypes(include=["category", "object", "string"])
    .columns
    .tolist()
)

datetime_columns = (
    X_train
    .select_dtypes(include=["datetime", "datetimetz"])
    .columns
    .tolist()
)

INTENTIONALLY_PRESERVED_COLUMNS = [
    "emp_length",
]

# -----------------------------------------------------------------------------
# 2. Numerical variables: median learned from X_train
# -----------------------------------------------------------------------------

numeric_columns_to_impute = [
    column
    for column in numeric_columns
    if (
        X_train[column].isna().any()
        or X_test[column].isna().any()
    )
]

numeric_medians = (
    X_train[numeric_columns_to_impute]
    .median()
)

X_train[numeric_columns_to_impute] = (
    X_train[numeric_columns_to_impute]
    .fillna(numeric_medians)
)

X_test[numeric_columns_to_impute] = (
    X_test[numeric_columns_to_impute]
    .fillna(numeric_medians)
)

# -----------------------------------------------------------------------------
# 3. Categorical variables: mode learned from X_train
# -----------------------------------------------------------------------------

categorical_columns_to_impute = [
    column
    for column in categorical_columns
    if (
        column not in INTENTIONALLY_PRESERVED_COLUMNS
        and (
            X_train[column].isna().any()
            or X_test[column].isna().any()
        )
    )
]

categorical_modes = {
    column: X_train[column].mode(dropna=True).iloc[0]
    for column in categorical_columns_to_impute
}

for column, mode_value in categorical_modes.items():
    X_train[column] = X_train[column].fillna(mode_value)
    X_test[column] = X_test[column].fillna(mode_value)

# -----------------------------------------------------------------------------
# 4. Datetime variables: median date learned from X_train
# -----------------------------------------------------------------------------

datetime_columns_to_impute = [
    column
    for column in datetime_columns
    if (
        X_train[column].isna().any()
        or X_test[column].isna().any()
    )
]

datetime_medians = {
    column: X_train[column].median()
    for column in datetime_columns_to_impute
}

for column, median_date in datetime_medians.items():
    X_train[column] = X_train[column].fillna(median_date)
    X_test[column] = X_test[column].fillna(median_date)

# -----------------------------------------------------------------------------
# 5. Validation
# -----------------------------------------------------------------------------

remaining_missing = pd.DataFrame({
    "X_train_missing": X_train.isna().sum(),
    "X_test_missing": X_test.isna().sum(),
})

remaining_missing = remaining_missing.loc[
    (remaining_missing["X_train_missing"] > 0)
    | (remaining_missing["X_test_missing"] > 0)
]

unexpected_missing = remaining_missing.drop(
    index=INTENTIONALLY_PRESERVED_COLUMNS,
    errors="ignore",
)

if not unexpected_missing.empty:
    raise ValueError(
        "Unexpected missing values remain after imputation:\n"
        f"{unexpected_missing}"
    )

print("Missing-value treatment completed successfully.")

if remaining_missing.empty:
    print("No missing values remain.")
else:
    print("Only intentionally preserved missing values remain:")
    display(remaining_missing)

Missing-value treatment completed successfully.
Only intentionally preserved missing values remain:


,X_train_missing,X_test_missing
emp_length,16715,4293


# Feature Screening and Final Dataset Construction

This section reviews the candidate predictors before model development.

The objective is to identify variables that should be retained, transformed or excluded based on data quality, business relevance and statistical properties.

The screening combines:

- unsupervised criteria, including quasi-constant variables, high-cardinality predictors and numerical correlation;
- supervised analyses of categorical variables based on the observed default rates in the development sample.

All transformation rules are learned exclusively from the development sample (`X_train`) and are subsequently applied unchanged to the test sample (`X_test`).

At the end of this section, two modelling datasets are produced:

- a Native Dataset;
- an Encoded Dataset.

### Initial Variable Screening

The first stage of feature screening provides a descriptive overview of all candidate predictors available for PD modelling.

Rather than making immediate selection decisions, this step summarizes the main characteristics of each variable, including:

- data type;
- percentage of missing values;
- number of distinct values (cardinality);
- proportion of unique values;
- frequency of the most common value.

These descriptive statistics provide an initial assessment of data quality and variable complexity. They serve as the basis for the subsequent review of constant variables, quasi-constant variables, high-cardinality predictors, business redundancy, and numerical correlations.

In [216]:
# =============================================================================
# 6.1 Initial Variable Screening
# =============================================================================

feature_screening = pd.DataFrame({
    "data_type": X_train.dtypes.astype(str),
    "missing_percentage": (
        X_train.isna().mean() * 100
    ).round(2),
    "unique_values": X_train.nunique(dropna=False),
    "unique_ratio": (
        X_train.nunique(dropna=False) / len(X_train)
    ).round(4),
    "most_frequent_ratio": X_train.apply(
        lambda column: column.value_counts(
            normalize=True,
            dropna=False
        ).iloc[0]
    ).round(4),
})

feature_screening = feature_screening.sort_values(
    by=[
        "most_frequent_ratio",
        "unique_values",
    ],
    ascending=[
        False,
        True,
    ],
)

feature_screening

,data_type,missing_percentage,unique_values,unique_ratio,most_frequent_ratio
acc_now_delinq,float64,0.0000,6,0.0000,0.9963
funding_ratio,float32,0.0000,1084,0.0029,0.9957
collections_12_mths_ex_med,float64,0.0000,9,0.0000,0.9917
emp_length_missing,Int8,0.0000,2,0.0000,0.9552
emp_title_missing,int8,0.0000,2,0.0000,0.9411
tot_coll_amt,float64,0.0000,5629,0.0151,0.8931
pub_rec,float64,0.0000,25,0.0001,0.8685
mths_since_last_record_is_missing,int8,0.0000,2,0.0000,0.8656
mths_since_last_record,float64,0.0000,124,0.0003,0.8656
investor_funding_ratio,float32,0.0000,11016,0.0295,0.8522


In [217]:
# =============================================================================
# 6.2 Initial screening decisions
# =============================================================================

QUASI_CONSTANT_THRESHOLD = 0.995
HIGH_CARDINALITY_THRESHOLD = 100

constant_features = feature_screening.index[
    feature_screening["unique_values"] <= 1
].tolist()

quasi_constant_features = feature_screening.index[
    (feature_screening["unique_values"] > 1)
    & (
        feature_screening["most_frequent_ratio"]
        >= QUASI_CONSTANT_THRESHOLD
    )
].tolist()

categorical_features = X_train.select_dtypes(
    include=["category", "object", "string"]
).columns

high_cardinality_features = [
    column
    for column in categorical_features
    if X_train[column].nunique(dropna=False)
    >= HIGH_CARDINALITY_THRESHOLD
]

print("Constant features:")
print(constant_features)

print("\nQuasi-constant features:")
print(quasi_constant_features)

print("\nHigh-cardinality categorical features:")
print(high_cardinality_features)

Constant features:
[]

Quasi-constant features:
['acc_now_delinq', 'funding_ratio']

High-cardinality categorical features:
['emp_title', 'title', 'zip_code']



The initial screening produced the following observations:

- no constant variables were identified;
- one quasi-constant variable (`acc_now_delinq`) was detected;
- three categorical variables (`emp_title`, `title`, and `zip_code`) exhibit high cardinality.

At this stage, no variable is removed automatically.

Although `acc_now_delinq` is highly unbalanced, it represents an important credit-risk characteristic and therefore requires a business review before any exclusion decision.

Similarly, high-cardinality variables are not discarded solely because of their number of distinct categories. Their relevance will be assessed individually during the business redundancy review and the categorical variable assessment.

## Business Redundancy Review

Several predictors describe the same underlying business information or have already been replaced by more informative engineered variables created in the Master Dataset.

These variables are reviewed before any statistical feature-selection procedure in order to avoid keeping multiple representations of the same information.

The following decisions are adopted:

- `grade` is removed because `sub_grade` provides a finer assessment of loan quality.
- `emp_length` is removed because its information is represented by the numerical variable `emp_length_years` together with the missing-value indicator `emp_length_missing`.
- `issue_d` is removed because the credit-age variable `credit_history_months` captures the relevant temporal information for PD modelling.
- `earliest_cr_line` is removed because its information is already incorporated into `credit_history_months`.

These decisions are based on business interpretation and feature engineering rather than on statistical criteria.

In [218]:
# =============================================================================
# 6.3 Business Redundancy Review
# =============================================================================

business_redundancy = pd.DataFrame({
    "original_variable": [
        "grade",
        "emp_length",
        "issue_d",
        "earliest_cr_line",
    ],
    "retained_variable": [
        "sub_grade",
        "emp_length_years + emp_length_missing",
        "credit_history_months",
        "credit_history_months",
    ],
    "decision": [
        "Remove",
        "Remove",
        "Remove",
        "Remove",
    ],
    "reason": [
        "sub_grade provides finer credit quality information.",
        "Engineered variables preserve employment length and missingness.",
        "Credit history duration is more informative than the loan issue date.",
        "Credit history duration replaces the original account opening date.",
    ],
})

display(business_redundancy)

,original_variable,retained_variable,decision,reason
0,grade,sub_grade,Remove,sub_grade provides finer credit quality inform...
1,emp_length,emp_length_years + emp_length_missing,Remove,Engineered variables preserve employment lengt...
2,issue_d,credit_history_months,Remove,Credit history duration is more informative th...
3,earliest_cr_line,credit_history_months,Remove,Credit history duration replaces the original ...


##  Numerical Correlation Analysis

The remaining numerical predictors are examined for linear dependence using the Pearson correlation coefficient.

Highly correlated variables may carry redundant information, leading to unnecessary model complexity and unstable coefficient estimates in generalized linear models.

Pairs of variables with an absolute correlation greater than or equal to **0.90** are identified for review. The final decision is based on both the strength of the correlation and the business interpretation of the variables.

No variable is removed automatically. The retained predictor is selected according to its business relevance, interpretability and potential usefulness for subsequent PD modelling.

In [219]:
# =============================================================================
# 6.4 Numerical Correlation Analysis
# =============================================================================

# Correlation threshold
CORRELATION_THRESHOLD = 0.90

# -----------------------------------------------------------------------------
# Select numerical variables
# -----------------------------------------------------------------------------

numeric_features = (
    X_train
    .select_dtypes(include=["number"])
    .copy()
)

# Remove binary indicator variables
numeric_features = numeric_features.loc[
    :,
    numeric_features.nunique() > 2
]

# -----------------------------------------------------------------------------
# Compute the Pearson correlation matrix
# -----------------------------------------------------------------------------

corr_matrix = numeric_features.corr(method="pearson")

# -----------------------------------------------------------------------------
# Extract highly correlated variable pairs
# -----------------------------------------------------------------------------

high_corr_pairs = []

columns = corr_matrix.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):

        corr = corr_matrix.iloc[i, j]

        if abs(corr) >= CORRELATION_THRESHOLD:

            high_corr_pairs.append(
                {
                    "Variable 1": columns[i],
                    "Variable 2": columns[j],
                    "Correlation": round(corr, 4),
                    "Absolute Correlation": round(abs(corr), 4),
                }
            )

high_corr_pairs_df = (
    pd.DataFrame(high_corr_pairs)
    .sort_values(
        by="Absolute Correlation",
        ascending=False,
    )
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# Display highly correlated pairs
# -----------------------------------------------------------------------------

print(
    f"Highly correlated variable pairs (|r| ≥ {CORRELATION_THRESHOLD})"
)

high_corr_pairs_df

Highly correlated variable pairs (|r| ≥ 0.9)


,Variable 1,Variable 2,Correlation,Absolute Correlation
0,loan_amnt,funded_amnt,0.9985,0.9985
1,funded_amnt,funded_amnt_inv,0.9960,0.9960
2,loan_amnt,funded_amnt_inv,0.9942,0.9942
3,investor_funding_ratio,investor_loan_ratio,0.9562,0.9562
4,funded_amnt,installment,0.9518,0.9518
5,loan_amnt,installment,0.9496,0.9496
6,funded_amnt_inv,installment,0.9472,0.9472
7,loan_to_income_ratio,installment_to_income_ratio,0.9399,0.9399


**Correlation Screening Decision**

The correlation analysis identified several groups of numerical variables carrying highly redundant information.

The following decisions were made by combining the observed correlations with the business meaning of each predictor:

- `loan_amnt` is retained as the primary loan-size variable, while `funded_amnt` and `funded_amnt_inv` are removed because they are almost perfectly correlated with it;
- `installment` is removed because it is largely determined by the loan amount, interest rate and term, while repayment burden remains represented by `installment_to_income_ratio`;
- `investor_loan_ratio` is retained, while `investor_funding_ratio` is removed because both variables contain nearly identical information;
- `installment_to_income_ratio` is retained as the more direct measure of periodic repayment burden, while `loan_to_income_ratio` is removed.

These variables are marked for removal but are not dropped from the datasets until all screening decisions have been completed.

In [220]:
# =============================================================================
# 6.4 Correlation Screening Decisions
# =============================================================================

correlated_features_to_drop = [
    "funded_amnt",
    "funded_amnt_inv",
    "installment",
    "investor_funding_ratio",
    "loan_to_income_ratio",
]

print("Features marked for removal after correlation review:")
print(correlated_features_to_drop)

Features marked for removal after correlation review:
['funded_amnt', 'funded_amnt_inv', 'installment', 'investor_funding_ratio', 'loan_to_income_ratio']


## Supervised Categorical Analysis

The predictive value of categorical variables is assessed by examining the default rate associated with each category.

Unlike the previous screening steps, this analysis incorporates the target variable (`pd_target`) to identify categories exhibiting similar default behaviour.

For ordinal variables, only adjacent categories may be considered for regrouping in order to preserve their natural ordering.

For nominal variables, category regrouping is considered only when it is supported by both business interpretation and similar observed default rates.

In [221]:
# =============================================================================
# Default rate by category
# =============================================================================

def categorical_default_summary(
    X,
    y,
    variable,
):
    summary = (
        pd.concat(
            [
                X[variable],
                y.rename("pd_target"),
            ],
            axis=1,
        )
        .groupby(variable, observed=False)
        .agg(
            observations=("pd_target", "count"),
            defaults=("pd_target", "sum"),
            default_rate=("pd_target", "mean"),
        )
        .sort_values("default_rate")
    )

    summary["default_rate"] = (
        summary["default_rate"] * 100
    ).round(2)

    return summary

In [222]:
categorical_variables = [
    "sub_grade",
    "purpose",
    "home_ownership",
    "verification_status",
    "addr_state",
]

In [223]:
for variable in categorical_variables:

    print("=" * 80)
    print(variable.upper())
    print("=" * 80)

    display(
        categorical_default_summary(
            X_train,
            y_train,
            variable,
        )
    )

SUB_GRADE


,observations,defaults,default_rate
sub_grade,,,
A1,8424,164,1.9500
A2,8765,259,2.9500
A3,10015,348,3.4700
A4,15208,654,4.3000
A5,17447,882,5.0600
B1,18347,1063,5.7900
B2,21278,1444,6.7900
B3,25278,1963,7.7700
B4,24287,2114,8.7000


PURPOSE


,observations,defaults,default_rate
purpose,,,
credit_card,83208,7100,8.5300
car,4244,386,9.1000
major_purchase,7856,757,9.6400
home_improvement,21206,2069,9.7600
debt_consolidation,219513,24930,11.3600
wedding,1857,215,11.5800
vacation,1983,240,12.1000
other,18988,2618,13.7900
medical,3724,515,13.8300


HOME_OWNERSHIP


,observations,defaults,default_rate
home_ownership,,,
MORTGAGE,188768,18098,9.5900
OWN,33290,3571,10.7300
RENT,150783,19067,12.6500
OTHER,187,38,20.3200


VERIFICATION_STATUS


,observations,defaults,default_rate
verification_status,,,
Not Verified,118645,11127,9.3800
Source Verified,120036,12509,10.4200
Verified,134347,17138,12.7600


ADDR_STATE


,observations,defaults,default_rate
addr_state,,,
ME,4,0,0.0000
WY,892,59,6.6100
DC,1153,82,7.1100
WV,1936,150,7.7500
NH,1817,143,7.8700
MS,973,81,8.3200
VT,731,64,8.7600
AK,965,85,8.8100
KS,3337,294,8.8100


**Final Decisions for Categorical Variables**

The supervised review of categorical predictors combines category frequencies, observed default rates and business interpretation to determine the final preprocessing strategy.

The following decisions are adopted:

- `sub_grade` is retained unchanged. The observed default rate increases almost monotonically from A1 to G5, confirming that the existing ordinal structure provides a meaningful representation of borrower credit quality.

- `purpose` is replaced by a broader grouped representation, `purpose_grouped`. Categories exhibiting similar default behaviour are combined while preserving an interpretable business structure. The original `purpose` variable is removed from the final PD dataset.

- `home_ownership` is retained unchanged. Its principal categories are sufficiently represented, have distinct business meanings and exhibit different default behaviours. No additional grouped variable is created.

- `verification_status` is retained unchanged because its categories are sufficiently represented and exhibit distinct default rates.

- `addr_state` is replaced by `addr_state_grouped`. States with fewer than 500 observations in the development sample are assigned to `OTHER_STATE`. States not observed during development are also assigned to `OTHER_STATE` when future observations are processed. The original `addr_state` variable is removed from the final PD dataset.

- `emp_title`, `title` and `zip_code` are excluded from the final PD modelling dataset because of their high cardinality and limited suitability for the conventional statistical models considered in this project.

The resulting categorical representation therefore retains meaningful original variables when appropriate and replaces only those variables for which a more stable or parsimonious representation has been justified.

All supervised decisions and grouping rules are derived exclusively from the development sample and are subsequently applied unchanged to the test sample.

## Final PD Dataset Construction

This section applies the feature-screening and transformation decisions established in the previous analyses.

All transformation rules that depend on the data are learned exclusively from the development sample and subsequently applied unchanged to the test sample.

The final construction process consists of:

1. creating the grouped representation of `purpose`;
2. grouping rare or previously unseen states into `OTHER_STATE`;
3. removing predictors excluded because of high cardinality, business redundancy, numerical redundancy, or replacement by engineered variables;
4. constructing the Native Dataset;
5. constructing the fully numerical Encoded Dataset.

No additional feature-selection decision is made at this stage. The purpose of this section is exclusively to apply the decisions already established and produce the final datasets for PD model development.

In [224]:
# =============================================================================
# 6.6.1 Purpose Grouping
# =============================================================================

X_train_final = X_train.copy()
X_test_final = X_test.copy()


PURPOSE_MAPPING = {
    "credit_card": "low_risk",
    "car": "low_risk",
    "major_purchase": "low_risk",
    "home_improvement": "low_risk",

    "debt_consolidation": "medium_risk",
    "wedding": "medium_risk",
    "vacation": "medium_risk",

    "other": "high_risk",
    "medical": "high_risk",
    "house": "high_risk",
    "moving": "high_risk",
    "renewable_energy": "high_risk",

    "educational": "very_high_risk",
    "small_business": "very_high_risk",
}


def apply_purpose_grouping(
    data: pd.DataFrame,
    mapping: dict,
) -> pd.DataFrame:
    """Apply the purpose-grouping rule defined on the development sample."""

    transformed = data.copy()

    transformed["purpose_grouped"] = (
        transformed["purpose"]
        .astype("string")
        .map(mapping)
        .fillna("other_or_unknown")
        .astype("category")
    )

    return transformed


X_train_final = apply_purpose_grouping(
    X_train_final,
    PURPOSE_MAPPING,
)

X_test_final = apply_purpose_grouping(
    X_test_final,
    PURPOSE_MAPPING,
)

### State Grouping

Rare-state identification is based exclusively on the development sample.

States containing fewer than 500 development observations are grouped into `OTHER_STATE`. States not observed during development are also assigned to this category when new observations are processed.

This rule improves statistical stability while preserving the information carried by sufficiently represented states.

In [225]:
# =============================================================================
# 6.6.2 State Grouping
# =============================================================================

MIN_STATE_OBSERVATIONS = 500

state_counts = (
    X_train["addr_state"]
    .astype("string")
    .value_counts()
)

known_states = set(state_counts.index)

rare_states = set(
    state_counts[
        state_counts < MIN_STATE_OBSERVATIONS
    ].index
)


def apply_state_grouping(
    data: pd.DataFrame,
    known_states: set,
    rare_states: set,
) -> pd.DataFrame:
    """Apply state-grouping rules learned from the development sample."""

    transformed = data.copy()

    states = (
        transformed["addr_state"]
        .astype("string")
    )

    grouped_states = states.where(
        states.isin(known_states)
        & ~states.isin(rare_states),
        "OTHER_STATE",
    )

    transformed["addr_state_grouped"] = (
        grouped_states
        .fillna("OTHER_STATE")
        .astype("category")
    )

    return transformed


X_train_final = apply_state_grouping(
    X_train_final,
    known_states,
    rare_states,
)

X_test_final = apply_state_grouping(
    X_test_final,
    known_states,
    rare_states,
)

print(
    "Rare states grouped:",
    sorted(rare_states),
)

Rare states grouped: ['IA', 'ID', 'ME', 'NE']


### Predictor Removal

All exclusion decisions established during feature screening are now applied simultaneously.

Variables are removed for one of four reasons:

- excessive categorical cardinality;
- business redundancy;
- strong numerical redundancy;
- replacement by a newly engineered representation.

Centralizing these exclusions at the final construction stage ensures that the analytical sections remain non-destructive and that all modelling decisions remain traceable.

In [226]:
# =============================================================================
# 6.6.3 Remove Excluded Predictors
# =============================================================================

FINAL_FEATURES_TO_DROP = [
    # -------------------------------------------------------------------------
    # High-cardinality predictors
    # -------------------------------------------------------------------------
    "emp_title",
    "title",
    "zip_code",

    # -------------------------------------------------------------------------
    # Business redundancy
    # -------------------------------------------------------------------------
    "grade",
    "emp_length",
    "issue_d",
    "earliest_cr_line",

    # -------------------------------------------------------------------------
    # Replaced categorical predictors
    # -------------------------------------------------------------------------
    "purpose",
    "addr_state",

    # -------------------------------------------------------------------------
    # Numerical redundancy
    # -------------------------------------------------------------------------
    "funded_amnt",
    "funded_amnt_inv",
    "installment",
    "investor_funding_ratio",
    "loan_to_income_ratio",
]


features_to_drop = [
    column
    for column in FINAL_FEATURES_TO_DROP
    if column in X_train_final.columns
]

X_train_final = X_train_final.drop(
    columns=features_to_drop
)

X_test_final = X_test_final.drop(
    columns=features_to_drop
)


print(
    f"Predictors removed: {len(features_to_drop)}"
)

print(features_to_drop)

Predictors removed: 14
['emp_title', 'title', 'zip_code', 'grade', 'emp_length', 'issue_d', 'earliest_cr_line', 'purpose', 'addr_state', 'funded_amnt', 'funded_amnt_inv', 'installment', 'investor_funding_ratio', 'loan_to_income_ratio']


### Native Dataset

The Native Dataset preserves the final predictors in their natural business representation.

Numerical predictors remain numerical, ordinal categorical variables preserve their existing ordering, and nominal categorical variables remain categorical.

This configuration can be used by modelling approaches capable of handling categorical predictors directly and also serves as the reference representation from which the encoded configuration is constructed.

In [227]:
# =============================================================================
# 6.6.4 Native Dataset
# =============================================================================

X_train_native = X_train_final.copy()
X_test_native = X_test_final.copy()


# Basic structural consistency
if not X_train_native.columns.equals(
    X_test_native.columns
):
    raise ValueError(
        "Native train and test datasets do not contain "
        "the same predictors."
    )


print(
    f"Native training dataset : "
    f"{X_train_native.shape[0]:,} rows × "
    f"{X_train_native.shape[1]} predictors"
)

print(
    f"Native test dataset     : "
    f"{X_test_native.shape[0]:,} rows × "
    f"{X_test_native.shape[1]} predictors"
)

Native training dataset : 373,028 rows × 40 predictors
Native test dataset     : 93,257 rows × 40 predictors


### Encoded Dataset

A fully numerical representation is derived from the Native Dataset for algorithms requiring numerical inputs.

Categorical predictors are separated according to their semantic structure:

- categorical variables with a genuine predefined ordering are transformed using ordinal encoding;
- nominal categorical variables are transformed using one-hot encoding;
- numerical variables are passed through unchanged.

The preprocessing structure is fitted exclusively on the development sample and subsequently applied unchanged to the test sample.

Previously unseen nominal categories are handled safely by the encoder, ensuring that the same transformation can later be applied to new loan applications.

In [228]:
# =============================================================================
# 6.6.5 Encoded Dataset
# =============================================================================

# -----------------------------------------------------------------------------
# Identify ordered categorical variables
# -----------------------------------------------------------------------------

ordered_categorical_columns = [
    column
    for column in X_train_native.select_dtypes(
        include="category"
    ).columns
    if X_train_native[column].cat.ordered
]


# -----------------------------------------------------------------------------
# Identify all remaining nominal categorical variables
# -----------------------------------------------------------------------------

all_categorical_columns = (
    X_train_native
    .select_dtypes(
        include=["category", "object", "string"]
    )
    .columns
    .tolist()
)

nominal_categorical_columns = [
    column
    for column in all_categorical_columns
    if column not in ordered_categorical_columns
]


# -----------------------------------------------------------------------------
# Identify numerical variables
# -----------------------------------------------------------------------------

numeric_columns = (
    X_train_native
    .select_dtypes(include="number")
    .columns
    .tolist()
)


# -----------------------------------------------------------------------------
# Preserve the business ordering already defined in the Master Dataset
# -----------------------------------------------------------------------------

ordinal_categories = [
    X_train_native[column]
    .cat.categories
    .tolist()
    for column in ordered_categorical_columns
]


# -----------------------------------------------------------------------------
# Build the preprocessing transformer
# -----------------------------------------------------------------------------

numerical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ordered_categorical_columns,
        ),
        (
            "nominal",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.int8,
            ),
            nominal_categorical_columns,
        ),
        (
            "numeric",
            "passthrough",
            numeric_columns,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
).set_output(
    transform="pandas"
)


# -----------------------------------------------------------------------------
# Fit on development sample only
# -----------------------------------------------------------------------------

X_train_encoded = (
    numerical_preprocessor
    .fit_transform(X_train_native)
)


# -----------------------------------------------------------------------------
# Apply unchanged transformation to test sample
# -----------------------------------------------------------------------------

X_test_encoded = (
    numerical_preprocessor
    .transform(X_test_native)
)


# Guarantee identical predictor ordering
X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns
)


# -----------------------------------------------------------------------------
# Basic structural consistency
# -----------------------------------------------------------------------------

if not X_train_encoded.columns.equals(
    X_test_encoded.columns
):
    raise ValueError(
        "Encoded train and test datasets do not contain "
        "the same predictors."
    )


non_numeric_columns = (
    X_train_encoded
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

if non_numeric_columns:
    raise TypeError(
        "Non-numerical predictors remain in the encoded dataset: "
        f"{non_numeric_columns}"
    )


print(
    f"Encoded training dataset : "
    f"{X_train_encoded.shape[0]:,} rows × "
    f"{X_train_encoded.shape[1]} predictors"
)

print(
    f"Encoded test dataset     : "
    f"{X_test_encoded.shape[0]:,} rows × "
    f"{X_test_encoded.shape[1]} predictors"
)

Encoded training dataset : 373,028 rows × 95 predictors
Encoded test dataset     : 93,257 rows × 95 predictors


## Final Validation and Export

The final step validates and exports the two PD dataset configurations created in the previous section:

- the **native configuration**, which preserves the remaining categorical variables;
- the **encoded configuration**, which contains only numerical predictors.

The validation confirms that:

- the training and testing samples contain aligned predictor columns;
- predictor and target indices remain aligned;
- no unexpected missing values remain;
- the encoded configuration contains only numerical variables;
- the target distribution remains consistent across the development and testing samples.

The validated datasets are then exported for use in the subsequent PD model-development notebooks.

In [229]:
# =============================================================================
# 7. Final Validation and Export
# =============================================================================

# -----------------------------------------------------------------------------
# 1. Validate dataset alignment
# -----------------------------------------------------------------------------

assert X_train_native.columns.equals(X_test_native.columns), (
    "Native train and test datasets do not contain the same columns."
)

assert X_train_encoded.columns.equals(X_test_encoded.columns), (
    "Encoded train and test datasets do not contain the same columns."
)

assert X_train_native.index.equals(y_train.index), (
    "X_train_native and y_train indices are not aligned."
)

assert X_test_native.index.equals(y_test.index), (
    "X_test_native and y_test indices are not aligned."
)

assert X_train_encoded.index.equals(y_train.index), (
    "X_train_encoded and y_train indices are not aligned."
)

assert X_test_encoded.index.equals(y_test.index), (
    "X_test_encoded and y_test indices are not aligned."
)


# -----------------------------------------------------------------------------
# 2. Validate missing values
# -----------------------------------------------------------------------------

native_train_missing = int(X_train_native.isna().sum().sum())
native_test_missing = int(X_test_native.isna().sum().sum())

encoded_train_missing = int(X_train_encoded.isna().sum().sum())
encoded_test_missing = int(X_test_encoded.isna().sum().sum())

assert native_train_missing == 0, (
    "Unexpected missing values remain in X_train_native."
)

assert native_test_missing == 0, (
    "Unexpected missing values remain in X_test_native."
)

assert encoded_train_missing == 0, (
    "Unexpected missing values remain in X_train_encoded."
)

assert encoded_test_missing == 0, (
    "Unexpected missing values remain in X_test_encoded."
)

# -----------------------------------------------------------------------------
# 4. Build final validation summary
# -----------------------------------------------------------------------------

validation_summary = pd.DataFrame({
    "dataset": [
        "X_train_native",
        "X_test_native",
        "X_train_encoded",
        "X_test_encoded",
    ],
    "observations": [
        X_train_native.shape[0],
        X_test_native.shape[0],
        X_train_encoded.shape[0],
        X_test_encoded.shape[0],
    ],
    "predictors": [
        X_train_native.shape[1],
        X_test_native.shape[1],
        X_train_encoded.shape[1],
        X_test_encoded.shape[1],
    ],
    "missing_values": [
        native_train_missing,
        native_test_missing,
        encoded_train_missing,
        encoded_test_missing,
    ],
})

display(validation_summary)


# -----------------------------------------------------------------------------
# 5. Validate target distributions
# -----------------------------------------------------------------------------

target_distribution = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
    "train_percentage": (
        y_train.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    ),
    "test_percentage": (
        y_test.value_counts(normalize=True)
        .sort_index()
        .mul(100)
    ),
}).round(4)

display(target_distribution)


# -----------------------------------------------------------------------------
# 7. Define export paths
# -----------------------------------------------------------------------------

export_paths = {
    "X_train_native": PD_DATA_DIR / "pd_X_train_native.parquet",
    "X_test_native": PD_DATA_DIR / "pd_X_test_native.parquet",
    "X_train_encoded": PD_DATA_DIR / "pd_X_train_encoded.parquet",
    "X_test_encoded": PD_DATA_DIR / "pd_X_test_encoded.parquet",
    "y_train": PD_DATA_DIR / "pd_y_train.parquet",
    "y_test": PD_DATA_DIR / "pd_y_test.parquet",
}


# -----------------------------------------------------------------------------
# 7. Export final datasets
# -----------------------------------------------------------------------------
# Indices are preserved because they guarantee alignment between X and y.

X_train_native.to_parquet(
    export_paths["X_train_native"],
    index=True,
)

X_test_native.to_parquet(
    export_paths["X_test_native"],
    index=True,
)

X_train_encoded.to_parquet(
    export_paths["X_train_encoded"],
    index=True,
)

X_test_encoded.to_parquet(
    export_paths["X_test_encoded"],
    index=True,
)

y_train.rename("pd_target").to_frame().to_parquet(
    export_paths["y_train"],
    index=True,
)

y_test.rename("pd_target").to_frame().to_parquet(
    export_paths["y_test"],
    index=True,
)


# -----------------------------------------------------------------------------
# 8. Confirm exported files
# -----------------------------------------------------------------------------

print("Final PD datasets exported successfully:\n")

for dataset_name, file_path in export_paths.items():
    print(f"{dataset_name:<18} -> {file_path}")

,dataset,observations,predictors,missing_values
0,X_train_native,373028,40,0
1,X_test_native,93257,40,0
2,X_train_encoded,373028,95,0
3,X_test_encoded,93257,95,0


,train_count,test_count,train_percentage,test_percentage
pd_target,,,,
0,332254,83063,89.0695,89.0689
1,40774,10194,10.9305,10.9311


Final PD datasets exported successfully:

X_train_native     -> C:\Users\htouy\OneDrive - HEC Montréal\Bureau\GitHub\risk-credit-scoring-new\data\processed\pd\pd_X_train_native.parquet
X_test_native      -> C:\Users\htouy\OneDrive - HEC Montréal\Bureau\GitHub\risk-credit-scoring-new\data\processed\pd\pd_X_test_native.parquet
X_train_encoded    -> C:\Users\htouy\OneDrive - HEC Montréal\Bureau\GitHub\risk-credit-scoring-new\data\processed\pd\pd_X_train_encoded.parquet
X_test_encoded     -> C:\Users\htouy\OneDrive - HEC Montréal\Bureau\GitHub\risk-credit-scoring-new\data\processed\pd\pd_X_test_encoded.parquet
y_train            -> C:\Users\htouy\OneDrive - HEC Montréal\Bureau\GitHub\risk-credit-scoring-new\data\processed\pd\pd_y_train.parquet
y_test             -> C:\Users\htouy\OneDrive - HEC Montréal\Bureau\GitHub\risk-credit-scoring-new\data\processed\pd\pd_y_test.parquet


In [230]:
# =============================================================================
# Audit of X_train_native
# =============================================================================

native_audit = pd.DataFrame({
    "data_type": X_train_native.dtypes.astype(str),
    "missing_count": X_train_native.isna().sum(),
    "missing_percentage": (
        X_train_native.isna().mean() * 100
    ).round(2),
    "unique_values": X_train_native.nunique(dropna=False),
})

native_audit.sort_values(
    by=["data_type", "missing_percentage"],
    ascending=[True, False],
)

,data_type,missing_count,missing_percentage,unique_values
credit_inquiry_rate,Float64,0,0.0000,2288
credit_history_months,Int16,0,0.0000,678
emp_length_years,Int8,0,0.0000,12
emp_length_missing,Int8,0,0.0000,2
sub_grade,category,0,0.0000,35
home_ownership,category,0,0.0000,4
verification_status,category,0,0.0000,3
initial_list_status,category,0,0.0000,2
purpose_grouped,category,0,0.0000,4
addr_state_grouped,category,0,0.0000,47


In [231]:
# =============================================================================
# Audit of X_train_encoded
# =============================================================================

encoded_audit = pd.DataFrame({
    "data_type": X_train_encoded.dtypes.astype(str),
    "missing_count": X_train_encoded.isna().sum(),
    "missing_percentage": (
        X_train_encoded.isna().mean() * 100
    ).round(2),
    "unique_values": X_train_encoded.nunique(dropna=False),
})

encoded_audit.sort_values(
    by=["data_type", "missing_percentage"],
    ascending=[True, False],
)

,data_type,missing_count,missing_percentage,unique_values
credit_inquiry_rate,Float64,0,0.0000,2288
credit_history_months,Int16,0,0.0000,678
emp_length_years,Int8,0,0.0000,12
emp_length_missing,Int8,0,0.0000,2
funding_ratio,float32,0,0.0000,1084
investor_loan_ratio,float32,0,0.0000,11370
sub_grade,float64,0,0.0000,35
int_rate,float64,0,0.0000,502
annual_inc,float64,0,0.0000,27062
dti,float64,0,0.0000,3997
